In [1]:
# ///////////////// Configuration /////////////////
learning_rate       = 0.02
inputMatrixSize     = (28, 28)
canvasSize          = (250, 250)

cnnLayersCount      = 2
fcnHiddenLayersSize = [64, 64, 32, 10]
filterSize          = [3, 3]
NUM_SAMPLES         = 70000

In [2]:
import numpy as np
from PIL import Image
from ipywidgets import Button, Label, FileUpload
from IPython.display import display
import io
from functools import reduce

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def ReLU(x):
    return max(0, x)
def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid_derivative(x):
    return x * (1 - x)

def predict(inputs):
    hidden = sigmoid(np.dot(inputs, syn0))
    output = sigmoid(np.dot(hidden, syn1))

    return output
from ipycanvas import Canvas

def create_draw_canvas():
    canvas = Canvas(width=canvasSize[0], height=canvasSize[1], sync_image_data=True)

    drawing = False
    last_x = 0
    last_y = 0
    
    canvas.stroke_style = "black"
    canvas.line_width = 10
    canvas.line_cap = "round"
    canvas.line_join = "round"
    
    def mouse_down(x, y):
        nonlocal drawing, last_x, last_y
        
        drawing = True
        last_x = x
        last_y = y
    
    def mouse_move(x, y):
        nonlocal last_x, last_y
        
        if drawing:
            canvas.begin_path()
            canvas.move_to(last_x, last_y)
            canvas.line_to(x, y)
            canvas.stroke()
            
            last_x = x
            last_y = y
    
    def mouse_up(x, y):
        nonlocal drawing
        drawing = False
    
    def mouse_out(x, y):
        nonlocal drawing
        drawing = False
    
    canvas.on_mouse_down(mouse_down)
    canvas.on_mouse_move(mouse_move)
    canvas.on_mouse_up(mouse_up)
    canvas.on_mouse_out(mouse_out)
    
    return canvas
def parse_matrix_canvas(canvas, matrix_size_x, matrix_size_y, canvas_size_x, canvas_size_y):
    pixels = np.array(canvas.get_image_data(0, 0, canvas_size_x, canvas_size_y))
    
    rgb = np.ones((canvas_size_x, canvas_size_y, 3), dtype=np.uint8) * 255
    
    alpha = pixels[:, :, 3] > 0
    rgb[alpha] = pixels[alpha, :3]
    
    gray = rgb.mean(axis=2)
    
    img = Image.fromarray(gray.astype(np.uint8))
    img = img.resize((matrix_size_x, matrix_size_y), Image.Resampling.LANCZOS)
    
    result = np.array(img)
    
    result = (result < 128).astype(int)
    return result

np.random.seed(1)

class FCN:

    def __init__(
        self,
        inputSize,
        hiddenLayersSizes,
        learningRate,
        model=None
    ):
        self.inputSize = inputSize
        self.layersSizes = [inputSize, *hiddenLayersSizes]
        self.learningRate = learningRate

        self.layers = [
            np.zeros(size, dtype=np.float32)
            for size in self.layersSizes
        ]

        if model is None:
            self.synapses = [
                (
                    np.random.randn(
                        self.layersSizes[l],
                        self.layersSizes[l + 1]
                    ).astype(np.float32)
                    *
                    np.sqrt(2.0 / self.layersSizes[l])
                )
                for l in range(len(self.layersSizes) - 1)
            ]
        else:
            self.synapses = model

    # =========================================================
    # Forward
    # =========================================================

    def request(self, I):

        self.layers[0] = I

        for l, W in enumerate(self.synapses):

            Z = np.dot(
                self.layers[l],
                W
            )

            Z = sigmoid(Z)

            self.layers[l + 1] = Z

        return self.layers[-1]

    # =========================================================
    # Evaluation
    # =========================================================

    def evaluate(self, data):

        correct = 0
        total_loss = 0.0

        for I, target in data:

            output = self.request(I)

            total_loss += np.mean(
                (target - output) ** 2
            )

            if np.argmax(output) == np.argmax(target):
                correct += 1

        return (
            total_loss / len(data),
            correct / len(data)
        )

    # =========================================================
    # Training
    # =========================================================

    def train_epoch(self, trainingData):

        n = len(trainingData)

        indices = np.random.permutation(n)

        d = [
            np.zeros_like(layer)
            for layer in self.layers
        ]

        for idx in indices:

            I, target = trainingData[idx]

            output = self.request(I)

            # -------------------------------------------------
            # Output delta
            # -------------------------------------------------

            d[-1] = (
                (output - target)
                *
                sigmoid_derivative(output)
            )

            # -------------------------------------------------
            # Hidden layers
            # -------------------------------------------------

            for l in range(
                len(self.layers) - 2,
                0,
                -1
            ):

                d[l] = (
                    np.dot(
                        self.synapses[l],
                        d[l + 1]
                    )
                    *
                    sigmoid_derivative(
                        self.layers[l]
                    )
                )

            # -------------------------------------------------
            # Gradient entering CNN
            # -------------------------------------------------

            d[0] = np.dot(
                self.synapses[0],
                d[1]
            )

            # -------------------------------------------------
            # Update
            # -------------------------------------------------

            for l in range(len(self.synapses)):

                self.synapses[l] -= (
                    self.learningRate
                    *
                    np.outer(
                        self.layers[l],
                        d[l + 1]
                    )
                )

        return d


class CNN:

    is_traning = False

    def __init__(
        self,
        inputMatrixSize,
        cnnLayersCount,
        fcnHiddenLayersSize,
        filterSize,
        learningRate,
        model=None
    ):

        self.inputMatrixSize = inputMatrixSize
        self.filterSize = tuple(filterSize)
        self.learningRate = learningRate

        # =====================================================
        # Layer sizes
        # =====================================================

        self.convolutionLayerSizes = [
            inputMatrixSize
        ]

        for _ in range(cnnLayersCount):

            self.convolutionLayerSizes.append(
                self.convolution_output_size(
                    self.convolutionLayerSizes[-1]
                )
            )

        # =====================================================
        # Layers
        # =====================================================

        self.layers = [
            np.zeros(
                size,
                dtype=np.float32
            )
            for size in self.convolutionLayerSizes
        ]

        # =====================================================
        # Model
        # =====================================================

        if model is None:

            self.fcn = FCN(
                np.prod(
                    self.convolutionLayerSizes[-1]
                ),
                fcnHiddenLayersSize,
                learningRate
            )

            fan_in = (
                self.filterSize[0]
                *
                self.filterSize[1]
            )

            self.synapses = [
                (
                    np.random.randn(
                        self.filterSize[0],
                        self.filterSize[1]
                    ).astype(np.float32)
                    *
                    np.sqrt(2.0 / fan_in)
                )
                for _ in range(
                    len(self.convolutionLayerSizes) - 1
                )
            ]

        else:

            self.fcn = FCN(
                np.prod(
                    self.convolutionLayerSizes[-1]
                ),
                fcnHiddenLayersSize,
                learningRate,
                model[1]
            )

            self.synapses = model[0]

    # =========================================================
    # Output size
    # =========================================================

    def convolution_output_size(
        self,
        shape
    ):

        return (
            shape[0] - self.filterSize[0] + 1,
            shape[1] - self.filterSize[1] + 1
        )

    # =========================================================
    # Convolution forward
    # =========================================================

    def convolution(
        self,
        X,
        W
    ):

        fh, fw = self.filterSize

        oh = X.shape[0] - fh + 1
        ow = X.shape[1] - fw + 1

        # Windows:
        #
        # (oh, ow, fh, fw)
        #
        windows = np.lib.stride_tricks.sliding_window_view(
            X,
            (fh, fw)
        )

        # Все свёртки сразу
        Z = np.sum(
            windows * W,
            axis=(2, 3)
        )

        return np.maximum(
            Z,
            0
        ).astype(np.float32)

    # =========================================================
    # Convolution backward
    # =========================================================

    def convolution_backward(
        self,
        delta,
        W
    ):

        fh, fw = self.filterSize

        h = delta.shape[0] + fh - 1
        w = delta.shape[1] + fw - 1

        result = np.zeros(
            (h, w),
            dtype=np.float32
        )

        # Вместо циклов по каждому пикселю:
        # только fh * fw операций.

        for m in range(fh):
            for n in range(fw):

                result[
                    m:m + delta.shape[0],
                    n:n + delta.shape[1]
                ] += (
                    delta * W[m, n]
                )

        return result

    # =========================================================
    # Forward CNN
    # =========================================================

    def request_cnn(self, I):

        self.layers[0] = I

        for layerIndex, W in enumerate(
            self.synapses
        ):

            self.layers[layerIndex + 1] = (
                self.convolution(
                    self.layers[layerIndex],
                    W
                )
            )

        return self.layers[-1]

    # =========================================================
    # Full network request
    # =========================================================

    def request(self, I):

        O = self.request_cnn(I).flatten()

        return self.fcn.request(O)

    # =========================================================
    # Evaluation
    # =========================================================

    def evaluate(self, data):

        correct = 0
        total_loss = 0.0

        for I, target in data:

            output = self.request(I)

            total_loss += np.mean(
                (target - output) ** 2
            )

            if np.argmax(output) == np.argmax(target):
                correct += 1

        return (
            total_loss / len(data),
            correct / len(data)
        )

    # =========================================================
    # Training
    # =========================================================

    def train_epoch(
        self,
        trainingData,
        on_batch=None
    ):

        total_loss = 0.0
        correct = 0

        n = len(trainingData)

        indices = np.random.permutation(n)

        # =====================================================
        # Training samples
        # =====================================================

        for batch, idx in enumerate(
            indices,
            start=1
        ):

            I, target = trainingData[idx]

            # -------------------------------------------------
            # CNN forward
            # -------------------------------------------------

            O = self.request_cnn(I)

            # -------------------------------------------------
            # FCN forward + backward
            # -------------------------------------------------

            fcn_delta = self.fcn.train_epoch([
                (
                    O.flatten(),
                    target
                )
            ])

            fcnD0 = fcn_delta[0]

            output = self.fcn.layers[-1]

            # -------------------------------------------------
            # Loss
            # -------------------------------------------------

            loss = np.mean(
                (target - output) ** 2
            )

            total_loss += loss

            # -------------------------------------------------
            # Accuracy
            # -------------------------------------------------

            if (
                np.argmax(output)
                ==
                np.argmax(target)
            ):
                correct += 1

            # -------------------------------------------------
            # Gradient from FCN → CNN
            # -------------------------------------------------

            deltas = [
                np.zeros_like(layer)
                for layer in self.layers
            ]

            deltas[-1] = (
                fcnD0.reshape(
                    self.convolutionLayerSizes[-1]
                )
                *
                relu_derivative(
                    self.layers[-1]
                )
            )

            # -------------------------------------------------
            # CNN backward
            # -------------------------------------------------

            for j in range(
                len(self.layers) - 2,
                0,
                -1
            ):

                # Equivalent to convolution with
                # flipped kernel.

                W = np.flip(
                    self.synapses[j],
                    (0, 1)
                )

                error = self.convolution_backward(
                    deltas[j + 1],
                    W
                )

                deltas[j] = (
                    error
                    *
                    relu_derivative(
                        self.layers[j]
                    )
                )

            # -------------------------------------------------
            # CNN weight gradients
            # -------------------------------------------------

            for layerIndex, W in enumerate(
                self.synapses
            ):

                X = self.layers[layerIndex]

                delta = deltas[
                    layerIndex + 1
                ]

                fh, fw = self.filterSize

                # Получаем все окна X
                windows = (
                    np.lib.stride_tricks
                    .sliding_window_view(
                        X,
                        (fh, fw)
                    )
                )

                # Градиент всего фильтра сразу
                gradient = np.einsum(
                    'ij,ijmn->mn',
                    delta,
                    windows
                )

                # Gradient descent
                W -= (
                    self.learningRate
                    *
                    gradient
                )

            # -------------------------------------------------
            # Callback
            # -------------------------------------------------

            if on_batch is not None:

                on_batch(
                    batch,
                    total_loss / batch,
                    correct
                )

        return (
            total_loss / n,
            correct / n
        )

In [3]:
import asyncio
import numpy as np
from sklearn.datasets import fetch_openml
import traceback
from math import prod
from concurrent.futures import ThreadPoolExecutor

# ====================== TRAINING ON THE BASIS OF MNIST ======================
print("Загружаю полный MNIST...")

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(int)

trainingData = []

def update_training_data(num_samples=1000):
    global trainingData

    indices = np.random.choice(
        len(X),
        num_samples,
        replace=False
    )

    trainingData = []

    for i in indices:
        image = X[i].reshape(
            inputMatrixSize[0],
            inputMatrixSize[1]
        )

        expected = np.zeros(
            10,
            dtype=np.float32
        )
        expected[y[i]] = 1.0

        trainingData.append([
            image,
            expected
        ])

print(f"Загружено {len(trainingData)} примеров")

# ====================== UI ======================
currentNumber = Label()
answer = Label()
batchInfo = Label()
canvas = create_draw_canvas()

buttonRecognize = Button(description="Recognize")
buttonClear     = Button(description="Clear")
buttonStart     = Button(description="Start")
buttonStop      = Button(description="Stop")

losses = []
is_training = False

cnn = CNN(inputMatrixSize, cnnLayersCount, fcnHiddenLayersSize, filterSize, learning_rate)

def on_batch(batch, loss, correct):
    if batch % 100 == 0:
        batchInfo.value = f"Batch: {batch}/{NUM_SAMPLES} " + f"({batch / NUM_SAMPLES * 100:.1f}%) " + f"Loss: {float(loss):.6f} " + f"Erroneous: {batch - correct}"


def training_loop():
    global is_training

    try:
        epoch = 0

        currentNumber.value = ">>> training_loop REALLY STARTED <<<"
        answer.value = "Первая эпоха начинается..."

        while is_training:
            update_training_data(NUM_SAMPLES)
            
            epoch += 1

            currentNumber.value = f">>> Epoch {epoch} running <<<"

            loss, correct = cnn.train_epoch(
                trainingData,
                on_batch
            )

            losses.append(loss)

            currentNumber.value = f"Epoch {epoch} finished"
            answer.value = (
                f"Epoch {epoch} | "
                f"Final Loss: {float(loss):.6f}"
            )

    except Exception as e:
        error_msg = (
            f"ОШИБКА:\n"
            f"{type(e).__name__}: {e}\n\n"
            f"{traceback.format_exc()}"
        )

        currentNumber.value = "!!! ОШИБКА !!!"
        answer.value = error_msg

        print(error_msg)

        is_training = False

executor = ThreadPoolExecutor(max_workers=1)

def start_training(b):
    global is_training

    if not is_training:
        is_training = True

        currentNumber.value = "Запускаю задачу..."
        answer.value = "Training thread запущен"

        executor.submit(training_loop)

def stop_training(b):
    global is_training

    is_training = False

    currentNumber.value = "Stopping..."
    
def clear_canvas(b):
    global canvas
    canvas.clear()

def recognize(b):
    try:
        data = (
            parse_matrix_canvas(
                canvas,
                inputMatrixSize[0],
                inputMatrixSize[1],
                canvasSize[0],
                canvasSize[1]
            )
            .astype(np.float32)
        )

        print(data)

        output = cnn.request(data)
        pred = int(np.argmax(output))
        conf = float(output[pred])

        answer.value = (
            f"Predicted: {pred} "
            f"(confidence: {conf:.3f})"
        )

    except Exception as e:
        answer.value = f"Ошибка Recognize: {e}"
        print(traceback.format_exc())

Загружаю полный MNIST...
Загружено 0 примеров


In [4]:
buttonRecognize.on_click(recognize)
buttonStart.on_click(start_training)
buttonStop.on_click(stop_training)
buttonClear.on_click(clear_canvas)

currentNumber.value = "FULL MNIST (70k) ready. Press Start"
answer.value = "—"

display(currentNumber)

display(answer)
display(batchInfo)
display(canvas)
display(buttonRecognize)
display(buttonClear)
display(buttonStart)
display(buttonStop)

Label(value='FULL MNIST (70k) ready. Press Start')

Label(value='—')

Label(value='')

Canvas(height=250, sync_image_data=True, width=250)

Button(description='Recognize', style=ButtonStyle())

Button(description='Clear', style=ButtonStyle())

Button(description='Start', style=ButtonStyle())

Button(description='Stop', style=ButtonStyle())

In [5]:
buttonLoadSynapses     = Button(description="Load Synapses")
buttonSaveSynapses     = Button(description="Save Synapses")

def load_synapses(b):
    global cnn
    data = np.load("model.npz", allow_pickle=True)

    cnn_model = data["cnn"]
    fcn_model = data["fcn"]

    cnn = CNN(
        inputMatrixSize,
        cnnLayersCount,
        fcnHiddenLayersSize,
        filterSize,
        learning_rate,
        [cnn_model, fcn_model]
    )
    print("Loaded")
def save_synapses(b):
    global cnn

    np.savez(
        "model.npz",
        cnn=np.array(cnn.synapses, dtype=object),
        fcn=np.array(cnn.fcn.synapses, dtype=object)
    )
    print("Saved")

buttonLoadSynapses.on_click(load_synapses)
buttonSaveSynapses.on_click(save_synapses)

display(buttonLoadSynapses)
display(buttonSaveSynapses)

Button(description='Load Synapses', style=ButtonStyle())

Button(description='Save Synapses', style=ButtonStyle())